# Training the "Hey Claude" wake-word model

Trains a microWakeWord model for the phrase "Hey Claude", for the Hey Claude
device firmware. Adapted from microWakeWord's `basic_training_notebook.ipynb`
(Apache License 2.0), with synthetic voices chosen for a range of accents,
sound-alike phrases as negatives, and progress saved to Google Drive.

The procedure, and what to do with the result, are described in
`docs/WAKEWORD-TRAINING.md` in the repository.

**Before starting:** Runtime → Change runtime type → a GPU (T4 on the free plan).
Run the cells in order. The first cell asks to restart the session once; after
that, continue from the second cell.

**Time:** sample generation and feature preparation take roughly an hour;
training takes one to several hours on a free T4. Keep the tab open; if the
session disconnects, rerun the cells up to cell 10, then restart the session
and run cell 2 and cell 11, as cell 11 explains. Generated samples and
training checkpoints are kept in Google Drive. Training starts from the
weights of its last checkpoint, but runs all of its steps again.

**Licence of the result:** the background-noise datasets downloaded below carry
mixed licences, some non-commercial, and the L2-ARCTIC voice is trained on
recordings licensed CC BY-NC 4.0. A model trained with them is for
non-commercial use.

**Privacy:** this notebook uses synthetic voices only. If real recordings are
added later, they are personal data: keep them in your own Drive, never in the
repository.

In [ ]:
# 1. Install the tools. microWakeWord, the sample generator and the two
# audio libraries are pinned; the rest is whatever Colab provides.
# When this finishes, use Runtime -> Restart session, then continue with cell 2.
# The last install fetches PyTorch and can take 5-15 minutes; it shows its
# progress so a long wait can be told apart from a stall.

MICROWAKEWORD_COMMIT = "4665173cd35f1cff9a61e06fc427f124766c488e"

# audio-metadata from a fork that unpins `attrs`, as microWakeWord's own notebook does
!pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
!git clone -q https://github.com/kahrendt/microWakeWord && cd microWakeWord && git checkout -q {MICROWAKEWORD_COMMIT}
!cd /content && pip install -q -e ./microWakeWord
!pip install piper-sample-generator==3.2.0
# The 3.2.0 package leaves out `piper_train`, which the US English generator
# voice needs to load. It is plain PyTorch code; the source at the same
# version supplies it (made importable in cell 2).
!git clone -q --depth 1 --branch v3.2.0 https://github.com/rhasspy/piper-sample-generator /content/piper-sample-generator
# The generator pins audiomentations to 0.33.0, which lacks AddColorNoise that
# microWakeWord's augmentation uses. Only the generator's optional augment tool
# uses audiomentations, and this notebook does not run it, so the newer version
# is installed over the pin. pip warns about the conflict; that is expected.
!pip install -q audiomentations==0.43.1

In [ ]:
# 2. Settings, and Google Drive for everything worth keeping between sessions.
# Large downloads stay on the session's own disk; they are fetched again if lost.

import importlib.util
import os
from google.colab import drive

# Cell 1 installs quietly, so a failed install would otherwise surface only in
# cell 7. Rerun its microWakeWord line without -q to see why, then restart.
if importlib.util.find_spec('microwakeword') is None:
    raise RuntimeError('microWakeWord is not installed: run cell 1 again, then Runtime -> Restart session')

drive.mount('/content/drive')

# Lets the sample generator, run as a separate program below, find
# `piper_train` (see cell 1)
os.environ['PYTHONPATH'] = '/content/piper-sample-generator:' + os.environ.get('PYTHONPATH', '')
KEEP = '/content/drive/MyDrive/hey_claude_wakeword'   # samples, checkpoints, results
os.makedirs(KEEP, exist_ok=True)
os.chdir('/content')

SAMPLES = f'{KEEP}/samples'

# Each training run keeps its checkpoints and results under its own name, so a
# new run starts fresh and an earlier one stays for comparison. Run history is
# in docs/WAKEWORD-TRAINING.md.
RUN = 'run2'
TRAIN_DIR = f'{KEEP}/trained_models/hey_claude_{RUN}'

# The wake phrase, spelled for each kind of voice. English voices read it as
# written; the Russian voices read a transliteration, which gives "Hey Claude"
# with a Russian accent. (Piper's Hebrew voice would add another accent, but it
# needs a newer Piper than the sample generator supports.)
PHRASE_EN = 'hey claude'
PHRASE_RU = 'хэй клод'

# Phrases that sound like the wake phrase but must not trigger it. "claude" and
# "hey" alone matter most: people near the device will talk about Claude.
SOUND_ALIKES = [
    'claude', 'hey', 'hey cloud', 'hey clyde', 'hey claw', 'hey clod', 'hey lord',
    'hey maud', 'hey claudia', 'okay claude', 'hi claude', 'say claude', 'hey cod',
    'hey clause', 'hey glad', 'they clawed', 'a cloud', 'hey jarvis', 'claude said',
    'ask claude',
]

In [ ]:
# 3. Voices. The US English "generator" mixes 904 LibriTTS-R speakers; the other
# voices are ordinary Piper voices that add accents the generator lacks.

import urllib.request

os.makedirs('voices', exist_ok=True)

def fetch(url, path):
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)

fetch('https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
      'voices/en_US-libritts_r-medium.pt')
# Its settings file is not in the release; it is in the source cloned in cell 1
import shutil
shutil.copy('/content/piper-sample-generator/models/en_US-libritts_r-medium.pt.json', 'voices/')

HF = 'https://huggingface.co/rhasspy/piper-voices/resolve/main'
VOICES = {
    # 24 speakers of English as a second language (L2-ARCTIC): Arabic, Mandarin,
    # Hindi, Korean, Spanish and Vietnamese first languages
    'l2arctic': 'en/en_US/l2arctic/medium/en_US-l2arctic-medium',
    # 109 speakers of British and other English accents (VCTK)
    'vctk': 'en/en_GB/vctk/medium/en_GB-vctk-medium',
    'ru_denis': 'ru/ru_RU/denis/medium/ru_RU-denis-medium',
    'ru_dmitri': 'ru/ru_RU/dmitri/medium/ru_RU-dmitri-medium',
    'ru_irina': 'ru/ru_RU/irina/medium/ru_RU-irina-medium',
    'ru_ruslan': 'ru/ru_RU/ruslan/medium/ru_RU-ruslan-medium',
}
for name, path in VOICES.items():
    base = path.split('/')[-1]
    fetch(f'{HF}/{path}.onnx', f'voices/{base}.onnx')
    fetch(f'{HF}/{path}.onnx.json', f'voices/{base}.onnx.json')

def voice(name):
    return 'voices/' + VOICES[name].split('/')[-1] + '.onnx'

In [ ]:
# 4. Listen before generating thousands. Each line plays one sample. If a voice
# mispronounces the phrase, try another spelling in cell 2 (for example
# 'hey clawd'), or drop that voice from cell 5.

from IPython.display import Audio, display

def sample(text, model, out):
    # The generator's messages are shown, so a failure explains itself
    !python3 -m piper_sample_generator "{text}" --model {model} --max-samples 1 --output-dir {out}
    wav = f'{out}/0.wav'
    if os.path.exists(wav):
        display(Audio(filename=wav))
    else:
        print(f'No sample from {model}: see the messages above.')

sample(PHRASE_EN, 'voices/en_US-libritts_r-medium.pt', 'check/generator')
sample(PHRASE_EN, voice('l2arctic'), 'check/l2arctic')
sample(PHRASE_EN, voice('vctk'), 'check/vctk')
sample(PHRASE_RU, voice('ru_irina'), 'check/ru')

In [ ]:
# 5. Generate the wake-phrase samples and the sound-alikes, into Drive. A set
# already generated in an earlier session is kept. Speaking speeds vary from
# fast (0.8) to slow (1.3).

SPEEDS = '0.8 0.9 1.0 1.1 1.2 1.3'

def generate(name, text, models, count, batch=1, extra=''):
    out = f'{SAMPLES}/{name}'
    if os.path.exists(out) and len(os.listdir(out)) >= count:
        print(f'{name}: already generated')
        return
    model_args = ' '.join(f'--model {m}' for m in models)
    !python3 -m piper_sample_generator "{text}" {model_args} --max-samples {count} \
        --batch-size {batch} --length-scales {SPEEDS} --output-dir {out} {extra}
    print(f'{name}: {len(os.listdir(out))} samples')

# Wake phrase. The generator runs on the GPU in batches; --max-speakers 800
# avoids the last LibriTTS speakers, which have little training data and
# produce artefacts.
generate('positive/generator', PHRASE_EN, ['voices/en_US-libritts_r-medium.pt'], 15000,
         batch=100, extra='--max-speakers 800')
# Accented voices: some speakers mispronounce the phrase, so each is kept to
# about an eighth of the samples; enough for the accents, not enough to teach a
# sloppy version of the phrase
generate('positive/l2arctic', PHRASE_EN, [voice('l2arctic')], 1000)
generate('positive/vctk', PHRASE_EN, [voice('vctk')], 1000)
generate('positive/ru', PHRASE_RU, [voice(v) for v in ('ru_denis', 'ru_dmitri', 'ru_irina', 'ru_ruslan')], 600)

# Sound-alikes: one phrase per line in a file, cycled through
with open('sound_alikes.txt', 'w') as f:
    f.write('\n'.join(SOUND_ALIKES))
generate('negative/generator', 'sound_alikes.txt', ['voices/en_US-libritts_r-medium.pt'], 4000,
         batch=100, extra='--max-speakers 800')
generate('negative/l2arctic', 'sound_alikes.txt', [voice('l2arctic')], 1000)

In [ ]:
# 6. Background sound for augmentation: room echoes (MIT impulse responses),
# everyday sounds (one part of AudioSet) and music (FMA). Adapted from
# microWakeWord's notebook. Mixed licences: see the note at the top. Slow.

import datasets
import scipy
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Files are numbered rather than named after their source: current versions of
# the `datasets` library no longer report a clip's original path. A marker
# file is written when a step completes, so a step interrupted part-way is
# done again rather than skipped.

def complete(output_dir):
    return os.path.exists(os.path.join(output_dir, '.complete'))

def save_16k(dataset, output_dir):
    # Resampled to 16 kHz, the rate of the device's microphone
    os.makedirs(output_dir, exist_ok=True)
    dataset = dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
    count = 0
    for i, row in enumerate(tqdm(dataset)):
        # Decoded and resampled audio can stray just past +-1.0. Clipped,
        # because such a sample would otherwise wrap round to a full-scale
        # click of the opposite sign.
        audio = np.clip(row['audio']['array'], -1.0, 1.0)
        scipy.io.wavfile.write(os.path.join(output_dir, f'{i}.wav'), 16000,
                               (audio * 32767).astype(np.int16))
        count += 1
    # A download that produced nothing is a failure, not a finished step
    if count == 0:
        raise RuntimeError(f'{output_dir}: no audio was downloaded')
    open(os.path.join(output_dir, '.complete'), 'w').close()

if not complete('mit_rirs'):
    save_16k(datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses',
                                   split='train', streaming=True), 'mit_rirs')

# One of the 38 parts of AudioSet's balanced set, about 690 MB, as
# microWakeWord's notebook uses. The dataset is now stored as parquet files,
# which `datasets` reads directly.
if not complete('audioset_16k'):
    audioset = datasets.load_dataset('agkphysics/AudioSet',
                                     data_files={'train': 'data/bal_train/09.parquet'},
                                     split='train')
    save_16k(audioset.select_columns(['audio']), 'audioset_16k')

if not complete('fma_16k'):
    os.makedirs('fma', exist_ok=True)
    !wget -q -O fma/fma_xs.zip https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
    !cd fma && unzip -q -o fma_xs.zip
    save_16k(datasets.Dataset.from_dict({'audio': [str(i) for i in Path('fma/fma_small').glob('**/*.mp3')]}),
             'fma_16k')

print('Background sound ready:', {d: len(os.listdir(d)) - 1 for d in ('mit_rirs', 'audioset_16k', 'fma_16k')})

In [ ]:
# 7. Augmentation: each sample is mixed with background sound, given room echo,
# and varied in level and tone, so the model hears the phrase as a microphone
# across a room would. Settings from microWakeWord's notebook, except that the
# background is never louder than the voice (0 to 15 dB below it, against -5
# to 10 dB there): run 1 learned largely from clips where the phrase was
# barely audible, and detected only about half of real tries.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=0,
    background_max_snr_db=15,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

def clips_from(subdir):
    # Every .wav under the directory, from all voices; 10 % held back for
    # validation and testing
    return Clips(input_directory=f'{SAMPLES}/{subdir}', file_pattern='**/*.wav',
                 max_clip_duration_s=None, remove_silence=False,
                 random_split_seed=10, split_count=0.1)

positive_clips = clips_from('positive')
negative_clips = clips_from('negative')

# Listen to one augmented wake-phrase sample
from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip
save_clip(augmenter.augment_clip(positive_clips.get_random_clip()), 'augmented_clip.wav')
Audio(filename='augmented_clip.wav')

In [ ]:
# 8. Turn the augmented samples into the spectrogram features the model trains
# on: the same 40-band features the device computes.

from mmap_ninja.ragged import RaggedMmap

def make_features(clips, out_root):
    for split, split_name, repeat, slide in (('training', 'train', 2, 10),
                                             ('validation', 'validation', 1, 1),
                                             ('testing', 'test', 1, 1)):
        out_dir = os.path.join(out_root, split)
        if os.path.exists(os.path.join(out_dir, 'wakeword_mmap')):
            continue
        os.makedirs(out_dir, exist_ok=True)
        # slide_frames: training reuses each spectrogram shifted by one step
        # at a time, imitating the streaming model; testing runs the streaming
        # model itself, so needs none. microWakeWord's notebook also slides the
        # validation set; here it does not, because the ten copies made
        # validation run out of memory on the free plan (see cell 9), and
        # every held-back sample is still validated once.
        spectrograms = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                             slide_frames=slide, step_ms=10)
        RaggedMmap.from_generator(out_dir=os.path.join(out_dir, 'wakeword_mmap'),
                                  sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repeat),
                                  batch_size=100, verbose=True)

make_features(positive_clips, 'features/positive')
make_features(negative_clips, 'features/sound_alikes')

In [ ]:
# 9. Negative examples prepared by microWakeWord's author: general speech,
# a dinner party, and sound without speech, already as features. Slow.

from pathlib import Path

if not os.path.exists('negative_datasets'):
    os.mkdir('negative_datasets')
    for fname in ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']:
        !wget -q -O negative_datasets/{fname} https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/{fname}
        !unzip -q negative_datasets/{fname} -d negative_datasets
        # The zip is no longer needed once unpacked, and the disk is not large
        !rm negative_datasets/{fname}

# Memory. For each validation check, microWakeWord loads the validation sets
# into memory whole and copies them twice more while shuffling. The free
# plan's 12.7 GB is not enough for all of it, and training is killed at the
# first check (step 500). Two reductions:
#
# 1. These three sets are used for training only, not for validation or
#    testing. Validation still covers recall and sound-alikes through this
#    notebook's own held-back samples. Renamed, not deleted, so the change can
#    be undone.
for name in ('speech', 'dinner_party', 'no_speech'):
    for split in ('validation', 'testing'):
        path = f'negative_datasets/{name}/{split}'
        if os.path.exists(path):
            os.rename(path, path + '_unused')

# 2. The dinner-party recording that false detections per hour are measured on
#    (its validation part) is about 9.7 hours, cut into 116,000 overlapping
#    windows (one every 300 ms): 3.8 GB, and over 11 GB with the copies.
#    Measured on a free T4 session: training holds about 3.9 GB, and a
#    validation share of 40 % added 6 GB, which was too much. Validation
#    therefore uses 15 % (about 1.5 hours), about 3 GB in all. microWakeWord
#    computes the hours from what is present, so the rate stays correct but
#    coarse: one false detection in 1.5 hours reads as about 0.7 per hour, so
#    a checkpoint meets the 0.5 per hour target only with none. The final test
#    runs on the separate test part, about 5.3 hours, which it processes piece
#    by piece without the large copies.
import numpy as np
from mmap_ninja.ragged import RaggedMmap

VALIDATION_SHARE = 0.15
SMALL = 'negative_datasets/dinner_party_eval_small'
if not os.path.exists(f'{SMALL}/validation_ambient/part_mmap'):
    source = [RaggedMmap(str(m)) for m in Path('negative_datasets/dinner_party_eval/validation_ambient').glob('**/*_mmap')]
    clips = [m[i] for m in source for i in range(len(m))]
    budget = int(VALIDATION_SHARE * sum(c.shape[0] for c in clips))
    # The recording is stored as a few clips of about 50 minutes each, so the
    # last one taken is cut to fit; a cut clip is a shorter stretch of the
    # same recording
    kept, used = [], 0
    for clip in clips:
        take = min(clip.shape[0], budget - used)
        if take <= 0:
            break
        kept.append(np.array(clip[:take]))
        used += take
    os.makedirs(f'{SMALL}/validation_ambient', exist_ok=True)
    RaggedMmap.from_generator(out_dir=f'{SMALL}/validation_ambient/part_mmap',
                              sample_generator=iter(kept), batch_size=10, verbose=False)
    # The final test uses the recording's whole test part, by reference
    if not os.path.exists(f'{SMALL}/testing_ambient'):
        os.symlink(os.path.abspath('negative_datasets/dinner_party_eval/testing_ambient'),
                   f'{SMALL}/testing_ambient')
    print(f'Validation recording: {len(kept)} piece(s), {used / 100 / 3600:.2f} hours')

In [ ]:
# 10. Training settings. From microWakeWord's notebook, with three changes: the
# sound-alike phrases are added as negatives, weighted up because they are the
# likeliest false triggers; the best checkpoint is chosen to stay under 0.5
# false detections per hour (the project's target, KNOWN-ISSUES R1) and then
# catch as many wake phrases as possible; and training is longer, 20,000
# steps and then 10,000 at a tenth of the learning rate to settle, where the
# notebook's starter value of 10,000 left run 1 under-trained. Settings given
# as one-entry lists apply to both phases.

import yaml

def features(path, weight, truth, strategy, penalty=1.0):
    return {'features_dir': path, 'sampling_weight': weight, 'penalty_weight': penalty,
            'truth': truth, 'truncation_strategy': strategy, 'type': 'mmap'}

config = {
    'window_step_ms': 10,
    'train_dir': TRAIN_DIR,
    'features': [
        features('features/positive', 2.0, True, 'truncate_start'),
        features('features/sound_alikes', 5.0, False, 'random', penalty=2.0),
        features('negative_datasets/speech', 10.0, False, 'random'),
        features('negative_datasets/dinner_party', 10.0, False, 'random'),
        features('negative_datasets/no_speech', 5.0, False, 'random'),
        # Validation and testing only: estimates false detections per hour.
        # The reduced copy made in cell 9; see the memory note there.
        features('negative_datasets/dinner_party_eval_small', 0.0, False, 'split'),
    ],
    'training_steps': [20000, 10000],
    'positive_class_weight': [1],
    'negative_class_weight': [20],
    'learning_rates': [0.001, 0.0001],
    'batch_size': 128,
    'time_mask_max_size': [0],
    'time_mask_count': [0],
    'freq_mask_max_size': [0],
    'freq_mask_count': [0],
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.5,
    'minimization_metric': 'ambient_false_positives_per_hour',
    'maximization_metric': 'average_viable_recall',
}
with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)

In [ ]:
# 11. Train, then convert to the quantised streaming model the device runs, and
# test it. If the session was lost, starts from the weights of the last
# checkpoint in Drive, but runs all the training steps again.
# The model shape matches the stock models the firmware was built with
# (stride 3).
#
# Restart the session first (Runtime -> Restart session), then run cell 2
# and this cell only. The notebook itself holds about 2 GB after cells 7-8,
# which training needs; it reads everything it uses from disk.
#
# Training runs as a separate process with its log in train.log. This cell
# prints the process's memory whenever it changes by 0.5 GB, then how the
# process ended and the end of the log. Exit code 0 is success; -9 means
# Colab stopped it for using too much memory (see the note in cell 9). Long
# silences are normal: memory stays flat while training.

import subprocess, time, psutil

args = ['python', '-m', 'microwakeword.model_train_eval',
        '--training_config=training_parameters.yaml', '--train', '1', '--restore_checkpoint', '1',
        '--test_tf_nonstreaming', '0', '--test_tflite_nonstreaming', '0',
        '--test_tflite_nonstreaming_quantized', '0', '--test_tflite_streaming', '0',
        '--test_tflite_streaming_quantized', '1', '--use_weights', 'best_weights',
        'mixednet', '--pointwise_filters', '64,64,64,64', '--repeat_in_block', '1, 1, 1, 1',
        '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]', '--residual_connection', '0,0,0,0',
        '--first_conv_filters', '32', '--first_conv_kernel_size', '5', '--stride', '3']
with open('train.log', 'w') as log:
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    p, peak, last = psutil.Process(proc.pid), 0, 0
    while proc.poll() is None:
        try:
            rss = p.memory_info().rss / 2**30   # bytes to GB
        except psutil.NoSuchProcess:
            break
        peak = max(peak, rss)
        if abs(rss - last) > 0.5:
            print(f'{time.strftime("%H:%M:%S")}  training process: {rss:.1f} GB')
            last = rss
        time.sleep(2)
print('exit code:', proc.wait(), ' peak:', f'{peak:.1f} GB')
!tail -40 train.log

In [ ]:
# 12. Keep the results: the model, and the test report that the cutoff is
# chosen from. Both are copied to Drive and downloaded.

import shutil
from google.colab import files

results = f'{TRAIN_DIR}/tflite_stream_state_internal_quant'
shutil.copy(f'{results}/stream_state_internal_quant.tflite', f'{KEEP}/hey_claude_{RUN}.tflite')
shutil.copy(f'{results}/tflite_streaming_roc.txt', f'{KEEP}/hey_claude_{RUN}_report.txt')
files.download(f'{KEEP}/hey_claude_{RUN}.tflite')
files.download(f'{KEEP}/hey_claude_{RUN}_report.txt')

# The test results for the streaming quantised model: recall and false
# detections per hour at each cutoff
!ls {TRAIN_DIR}/tflite_stream_state_internal_quant/
for name in os.listdir(f'{TRAIN_DIR}/tflite_stream_state_internal_quant'):
    if name.endswith('.txt'):
        print(open(f'{TRAIN_DIR}/tflite_stream_state_internal_quant/{name}').read())